## 最終課題

In [ ]:
# webスクレイピングに最低限必要なライブラリをインポート
import requests
from bs4 import BeautifulSoup, Comment
from urllib.parse import urljoin
import time

In [15]:
url = "https://www.musashino-u.ac.jp/" #最初のURL
domain = "musashino-u.ac.jp" #ドメイン部分

to_visit = [url]
visited = set()
result = {}


In [42]:
while to_visit:
    url = to_visit.pop(0)
    if url in visited:
        continue

    try:
        response = requests.get(url)
        response.encoding = response.apparent_encoding
    except Exception as e:
        continue

    visited.add(url)
    soup = BeautifulSoup(response.text, "html.parser")

    # コメントアウト削除（コメント内リンク除外）
    for c in soup.find_all(text=lambda t: isinstance(t, Comment)):
        c.extract()

    # titleタグ取得（存在しない場合スキップ）
    title_tag = soup.find("title")
    if not title_tag or not title_tag.get_text().strip():
        continue

    title_text = title_tag.get_text().strip()
    result[url] = title_text

    # aタグ探索（相対パス含む）
    for a_tag in soup.find_all("a", href=True):
        link = a_tag["href"].strip()

        # 無効・不要なリンクを除外
        if not link or link in ["#", "/", " "]:
            continue
        if link.startswith(("javascript:", "mailto:", "tel:", "#")):
            continue
        if any(char in link for char in ["[", "]", " ", "<", ">"]):
            continue  # IPv6風や壊れURL対策

        # 相対パスを絶対URLへ変換
        full_link = urljoin(url, link)

        # 外部サイト・不要ファイル除外
        if domain not in full_link:
            continue
        if any(ext in full_link for ext in [".pdf", ".jpg", ".png", ".mp4", ".zip"]):
            continue

        if full_link not in visited and full_link not in to_visit:
            to_visit.append(full_link)

    # サーバーへの負荷軽減
    time.sleep(1)

    # 上限設定（安全のため）
    if len(result) >= 1800:
        print("1800ページに到達したため終了します。")
        break

/var/folders/ts/sk8mfy_x6cl0pg0hrmp0mfdr0000gn/T/ipykernel_99555/1713202587.py:16: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  for c in soup.find_all(text=lambda t: isinstance(t, Comment)):


1800ページに到達したため終了します。


In [43]:
print(result)

{'https://www.musashino-u.ac.jp/': '武蔵野大学', 'https://ef.musashino-u.ac.jp/donation/': 'ご寄付のお願い | 学校法人武蔵野大学', 'https://dl.musashino-u.ac.jp/': '武蔵野大学通信教育部', 'https://www.musashino-u.ac.jp/news/detail/20251029-7380.html': '2026年度に通信教育部国際データサイエンス学部を開設します | ニュース | 武蔵野大学', 'https://www.musashino-u.ac.jp/news/detail/20250701-6913.html': '2026年度にウェルビーイング研究科を開設します | ニュース | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/profile/message.html': '学長就任のごあいさつ | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/basic/buddhist_studies/': '副専攻（仏教プラクティスコース） | 武蔵野大学', 'https://sdgs.musashino-u.ac.jp/': '武蔵野大学×SDGs', 'https://www.musashino-u.ac.jp/guide/profile/infographic.html': '数字で見る武蔵野大学 | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/guide/profile/history.html': '武蔵野大学の歩み | 大学案内 | 武蔵野大学', 'https://www.musashino-u.ac.jp/international/study-abroad/index.html': '武蔵野大学から世界へ | 国際交流・留学 | 武蔵野大学', 'https://www.musashino-u.ac.jp/basic/initial/fs/index.html': 'フィールド・スタディーズ | 武蔵野大学', 'https://www.musashino-u.ac.jp/st